# NB37: MASTER Panel — Kalibre Stacking (Isotonic + Çeşitli Base Modeller)

**Amaç:** NB36'nın DOĞRU değerlendirme protokolünü (M3 missing, %80/20 bootstrap, benign-aware threshold) koruyup, NB14'ün kaybedilmiş iki modelleme kazanımını üzerine ekle:
1. **Isotonic Regresyon ile OOF olasılık kalibrasyonu** — Base modellerin tahminlerini meta-learner'a vermeden ÖNCE kalibre et.
2. **Model çeşitliliği:** Sadece ağaç modeli değil, en az 5 farklı base: CatBoost, LightGBM, XGBoost (veya BalancedBagging), LogisticRegression, MLP/kNN.

**Meta-learner = LogisticRegression** (L2, class_weight='balanced'). GBM/ağaç meta KULLANMA (overfit eder).

**FE ablasyon:** Varsayılan `no_fe` (NB36'da zararlı bulundu). `with_fe` ayrı kol; ikisini %80/20 ile yan yana raporla.

**Leakage robust:** `AL_MISSINGNESS_LEAKAGE_RISK` dahil/hariç iki model yan yana.

**Değerlendirme (NB36 AYNEN):** Threshold 3 modda (f1_raw, f1_8020, mcc_8020). Test hem %50/50 hem bootstrap %80/20 (N=50, %95 CI). Birincil = %80/20 pathogenic-F1.

---


In [1]:
# Cell 1: Imports ve Setup
import sys, os, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy
from collections import OrderedDict

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
    PROJECT_ROOT = os.path.dirname(os.getcwd())
if not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
    PROJECT_ROOT = os.getcwd()
    while PROJECT_ROOT != '/' and not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

from config import SEED, TEST_SIZE, PROJECT_ROOT, REPORTS_DIR
from src.columns_real import (
    ID_COL, TARGET_COL, NON_FEATURE_COLS,
    AL_COLS, CAT_COLS, EK_COLS, AA_COLS, ALL_FEATURE_COLS,
    AL_HIGH_MISSING_COLS, AL_MISSINGNESS_LEAKAGE_RISK, AL_SAFE_COLS,
    CAT_POPULATION_COLS, CAT_GENOTYPE_COLS, CAT_REGION_COLS,
    get_constant_cols, get_duplicate_col_pairs, get_missing_mask_col_name,
    AA_ALPHABET, AA_UNKNOWN_TOKEN
)
from src.metrics import optimize_threshold, compute_all_metrics

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.isotonic import IsotonicRegression
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix

import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb

import torch
import torch.nn as nn
from sklearn.neural_network import MLPClassifier

from fpdf import FPDF

print(f"PyTorch device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"SEED: {SEED}, TEST_SIZE: {TEST_SIZE}")

PyTorch device: cpu
PROJECT_ROOT: /Users/tefe/teknofest_model/teknofest_model
SEED: 42, TEST_SIZE: 0.2


In [2]:
# Cell 2: Veri Yükleme ve İlk Temizlik
MASTER_CSV = os.path.join(PROJECT_ROOT, 'data/real_data/YARISMA_TRAIN_MASTER.csv')
df = pd.read_csv(MASTER_CSV)

print(f"Veri yüklendi: {df.shape}")
print(f"Sütunlar: {df.columns.tolist()[:10]}...")
print(f"\nLabel dağılımı:\n{df[TARGET_COL].value_counts()}")
print(f"\nEksiklik oranı: {df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100:.2f}%")

Veri yüklendi: (2931, 353)
Sütunlar: ['Variant_ID', 'AL_1', 'AL_2', 'AL_3', 'AL_4', 'AL_5', 'AL_6', 'AL_7', 'AL_8', 'AL_9']...

Label dağılımı:
Label
1    2149
0     782
Name: count, dtype: int64

Eksiklik oranı: 54.94%


In [3]:
# Cell 3: Stratified Split (ÖNCE split, SONRA her şey)
feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
X = df[feature_cols].copy()
y = df[TARGET_COL].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

print(f"Train: {X_train.shape} (pos={y_train.sum()}, neg={(y_train==0).sum()})")
print(f"Test:  {X_test.shape} (pos={y_test.sum()}, neg={(y_test==0).sum()})")

Train: (2344, 351) (pos=1719, neg=625)
Test:  (587, 351) (pos=430, neg=157)


In [4]:
# Cell 4: Sabit ve Özdeş Sütun Temizliği (train üzerinde tespit)
const_cols = get_constant_cols(X_train)
dup_col_pairs = get_duplicate_col_pairs(X_train)
dup_cols = set()
for c1, c2 in dup_col_pairs:
    dup_cols.add(c2)  # biri drop et

drop_cols = const_cols + list(dup_cols)
drop_cols = list(set(drop_cols))  # uniq

print(f"Sabit sütun: {len(const_cols)}")
print(f"Özdeş çift: {len(dup_col_pairs)} → {len(dup_cols)} drop")
print(f"Toplam drop: {len(drop_cols)}")

# CAT_6 da drop (zaten high-missing olmalı)
if 'CAT_6' in X_train.columns and 'CAT_6' not in drop_cols:
    drop_cols.append('CAT_6')

feature_cols = [c for c in feature_cols if c not in drop_cols]
X_train = X_train[feature_cols].copy()
X_test = X_test[feature_cols].copy()

print(f"\nFeature sayısı sonra: {X_train.shape[1]}")

Sabit sütun: 57
Özdeş çift: 583 → 58 drop
Toplam drop: 63

Feature sayısı sonra: 287


In [5]:
# Cell 5: M3 Missing Stratejisi + Medyan Imputation
# >%50 NaN sütunlar için is_missing_* flag + tüm sayısala medyan imputation

missing_threshold = 0.50
missing_mask_cols = {}

for col in feature_cols:
    miss_ratio = X_train[col].isnull().sum() / len(X_train)
    if miss_ratio > missing_threshold:
        mask_col_name = get_missing_mask_col_name(col)
        missing_mask_cols[col] = mask_col_name

print(f">%50 missing sütun: {len(missing_mask_cols)}")

# Imputation (train üzerinde fit)
imputer = SimpleImputer(strategy='median')
X_train_imp = X_train.copy()
X_test_imp = X_test.copy()

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    X_train_imp[numeric_cols] = imputer.fit_transform(X_train[numeric_cols])
    X_test_imp[numeric_cols] = imputer.transform(X_test[numeric_cols])

# is_missing_* flag'leri ekle (train + test)
for orig_col, mask_col in missing_mask_cols.items():
    X_train_imp[mask_col] = X_train[orig_col].isnull().astype(int)
    X_test_imp[mask_col] = X_test[orig_col].isnull().astype(int)

print(f"\nNew features (flags dahil): {X_train_imp.shape[1]}")
print(f"Örnek flag sütunlar: {list(missing_mask_cols.values())[:5]}")

>%50 missing sütun: 139

New features (flags dahil): 426
Örnek flag sütunlar: ['is_missing_AL_1', 'is_missing_AL_2', 'is_missing_AL_3', 'is_missing_AL_4', 'is_missing_AL_5']


In [6]:
# Cell 6: Değerlendirme Yardımcı Fonksiyonları (NB36 ile BIREBIR — imza-uyumlu)
from sklearn.metrics import matthews_corrcoef, precision_score, recall_score

def optimize_threshold_8020(y_true, y_prob, n_bootstrap=50, target_pos_rate=0.20):
    """%80/20 dagilima yeniden ornekleinmis havuzda F1-max ve MCC-max threshold sec (NB36 birebir)."""
    best_thrs_f1, best_thrs_mcc = [], []
    rng = np.random.RandomState(SEED)
    idx_pos = np.where(y_true == 1)[0]
    idx_neg = np.where(y_true == 0)[0]
    for _ in range(n_bootstrap):
        n_neg = len(idx_neg)
        n_pos_target = max(1, int(n_neg * target_pos_rate / (1 - target_pos_rate)))
        if n_pos_target > len(idx_pos):
            n_pos_target = len(idx_pos)
        sel_pos = rng.choice(idx_pos, size=n_pos_target, replace=True)
        sel_neg = rng.choice(idx_neg, size=n_neg, replace=True)
        sel = np.concatenate([sel_pos, sel_neg])
        y_sel = y_true.values[sel] if hasattr(y_true, 'values') else y_true[sel]
        p_sel = y_prob[sel]
        best_f1, best_t_f1 = 0, 0.5
        best_mcc, best_t_mcc = -1, 0.5
        for thr in np.arange(0.10, 0.90, 0.01):
            preds = (p_sel >= thr).astype(int)
            f1 = f1_score(y_sel, preds, zero_division=0)
            mcc = matthews_corrcoef(y_sel, preds)
            if f1 > best_f1:
                best_f1, best_t_f1 = f1, thr
            if mcc > best_mcc:
                best_mcc, best_t_mcc = mcc, thr
        best_thrs_f1.append(best_t_f1)
        best_thrs_mcc.append(best_t_mcc)
    return np.median(best_thrs_f1), np.median(best_thrs_mcc)


def bootstrap_8020_eval(y_true, y_prob, threshold, n_bootstrap=50, target_pos_rate=0.20):
    """Bootstrap %80/20 dagilimda metrik hesapla (NB36 birebir). F1/MCC/prec/rec + CI."""
    rng = np.random.RandomState(SEED + 1)
    idx_pos = np.where(y_true == 1)[0]
    idx_neg = np.where(y_true == 0)[0]
    f1s, mccs, precs, recs = [], [], [], []
    for _ in range(n_bootstrap):
        n_neg = len(idx_neg)
        n_pos_target = max(1, int(n_neg * target_pos_rate / (1 - target_pos_rate)))
        if n_pos_target > len(idx_pos):
            n_pos_target = len(idx_pos)
        sel_pos = rng.choice(idx_pos, size=n_pos_target, replace=True)
        sel_neg = rng.choice(idx_neg, size=n_neg, replace=True)
        sel = np.concatenate([sel_pos, sel_neg])
        y_sel = y_true.values[sel] if hasattr(y_true, 'values') else y_true[sel]
        p_sel = y_prob[sel]
        preds = (p_sel >= threshold).astype(int)
        f1s.append(f1_score(y_sel, preds, zero_division=0))
        mccs.append(matthews_corrcoef(y_sel, preds))
        precs.append(precision_score(y_sel, preds, zero_division=0))
        recs.append(recall_score(y_sel, preds, zero_division=0))
    return {
        'f1_mean': float(np.mean(f1s)), 'f1_std': float(np.std(f1s)),
        'f1_ci_lo': float(np.percentile(f1s, 2.5)), 'f1_ci_hi': float(np.percentile(f1s, 97.5)),
        'f1_scores': f1s,
        'mcc_mean': float(np.mean(mccs)), 'prec_mean': float(np.mean(precs)), 'rec_mean': float(np.mean(recs)),
    }

print("Degerlendirme fonksiyonlari yuklendi (optimize_threshold_8020 + bootstrap_8020_eval, NB36 birebir).")


Degerlendirme fonksiyonlari yuklendi (optimize_threshold_8020 + bootstrap_8020_eval, NB36 birebir).


In [7]:
# Cell 7: Encoding Yardımcıları
def encode_cat_cols(X_train, X_test, cat_cols, encode_type='label'):
    """
    CAT sütunlarını encode et (train üzerinde fit).
    """
    X_train_enc = X_train.copy()
    X_test_enc = X_test.copy()
    
    if encode_type == 'label':
        for col in cat_cols:
            if col in X_train_enc.columns:
                le = LabelEncoder()
                # Fit on train (fill NaN s bir category olarak)
                train_vals = X_train_enc[col].fillna('__NA__').astype(str)
                le.fit(train_vals)
                X_train_enc[col] = le.transform(train_vals)
                test_vals = X_test_enc[col].fillna('__NA__').astype(str)
                X_test_enc[col] = le.transform(test_vals)
    elif encode_type == 'drop':
        X_train_enc = X_train_enc.drop(columns=[c for c in cat_cols if c in X_train_enc.columns])
        X_test_enc = X_test_enc.drop(columns=[c for c in cat_cols if c in X_test_enc.columns])
    
    return X_train_enc, X_test_enc

print("Encoding fonksiyonları hazır.")

Encoding fonksiyonları hazır.


In [8]:
# Cell 8: Base Model 1 — LightGBM
def train_lgbm(X_train, X_test, y_train, y_test, trial_id=""):
    """
    LightGBM eğit ve OOF probability'leri döndür.
    """
    # Train üzerinde OOF oluştur (StratifiedKFold 5)
    oof_proba_train = np.zeros(len(X_train))
    oof_proba_test = np.zeros(len(X_test))
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_models = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = lgb.LGBMClassifier(
            n_estimators=100,
            learning_rate=0.1,
            random_state=SEED,
            verbose=-1
        )
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(10)])
        
        oof_proba_train[val_idx] = model.predict_proba(X_val)[:, 1]
        oof_proba_test += model.predict_proba(X_test)[:, 1] / 5
        fold_models.append(model)
    
    return oof_proba_train, oof_proba_test, fold_models

print("LightGBM trainer hazır.")

LightGBM trainer hazır.


In [9]:
# Cell 9: Base Model 2 — CatBoost (native categorical)

CAT_MISSING_TOKEN = "__CAT_MISSING__"


def _prepare_catboost_native(X_train, X_test, cat_cols=None):
    """CatBoost icin kategorikleri string/sentinel, numerikleri float yap."""
    cat_cols = [c for c in (cat_cols or []) if c in X_train.columns]
    cat_set = set(cat_cols)
    X_tr = X_train.copy()
    X_te = X_test.copy()

    for col in X_tr.columns:
        if col in cat_set:
            X_tr[col] = X_tr[col].astype('object').where(X_tr[col].notna(), CAT_MISSING_TOKEN).astype(str)
            X_te[col] = X_te[col].astype('object').where(X_te[col].notna(), CAT_MISSING_TOKEN).astype(str)
        else:
            tr_num = pd.to_numeric(X_tr[col], errors='coerce')
            te_num = pd.to_numeric(X_te[col], errors='coerce')
            fill_value = tr_num.median()
            if pd.isna(fill_value):
                fill_value = 0.0
            X_tr[col] = tr_num.fillna(fill_value).astype(float)
            X_te[col] = te_num.fillna(fill_value).astype(float)

    return X_tr, X_te, cat_cols


def train_catboost(X_train, X_test, y_train, y_test, cat_cols=None, trial_id=""):
    """CatBoost OOF probability; native cat kolonlar NaN icermeyecek sekilde hazirlanir."""
    X_cb_train, X_cb_test, cat_cols = _prepare_catboost_native(X_train, X_test, cat_cols)
    cat_features = cat_cols if cat_cols else []

    oof_proba_train = np.zeros(len(X_cb_train))
    oof_proba_test = np.zeros(len(X_cb_test))

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_models = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_cb_train, y_train)):
        X_tr, X_val = X_cb_train.iloc[train_idx], X_cb_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = CatBoostClassifier(
            iterations=100,
            learning_rate=0.1,
            random_state=SEED,
            cat_features=cat_features,
            verbose=False
        )
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])

        oof_proba_train[val_idx] = model.predict_proba(X_val)[:, 1]
        oof_proba_test += model.predict_proba(X_cb_test)[:, 1] / 5
        fold_models.append(model)

    return oof_proba_train, oof_proba_test, fold_models


print("CatBoost trainer hazır (NaN-safe native categorical).")


CatBoost trainer hazır (NaN-safe native categorical).


In [10]:
# Cell 10: Base Model 3 — XGBoost
def train_xgboost(X_train, X_test, y_train, y_test, trial_id=""):
    """
    XGBoost OOF probability.
    """
    oof_proba_train = np.zeros(len(X_train))
    oof_proba_test = np.zeros(len(X_test))
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_models = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = xgb.XGBClassifier(
            n_estimators=100,
            learning_rate=0.1,
            random_state=SEED,
            verbosity=0
        )
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        
        oof_proba_train[val_idx] = model.predict_proba(X_val)[:, 1]
        oof_proba_test += model.predict_proba(X_test)[:, 1] / 5
        fold_models.append(model)
    
    return oof_proba_train, oof_proba_test, fold_models

print("XGBoost trainer hazır.")

XGBoost trainer hazır.


In [11]:
# Cell 11: Base Model 4 — LogisticRegression (L1/L2)
def train_logistic(X_train, X_test, y_train, y_test, penalty='l2', trial_id=""):
    """
    LogisticRegression OOF (L1 veya L2).
    """
    # Scale (LR için zorunlu)
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)
    
    oof_proba_train = np.zeros(len(X_train))
    oof_proba_test = np.zeros(len(X_test))
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_models = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_val = X_train_sc[train_idx], X_train_sc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = LogisticRegression(
            penalty=penalty,
            max_iter=1000,
            random_state=SEED,
            class_weight='balanced'
        )
        model.fit(X_tr, y_tr)
        
        oof_proba_train[val_idx] = model.predict_proba(X_val)[:, 1]
        oof_proba_test += model.predict_proba(X_test_sc)[:, 1] / 5
        fold_models.append(model)
    
    return oof_proba_train, oof_proba_test, fold_models

print("LogisticRegression trainer hazır.")

LogisticRegression trainer hazır.


In [12]:
# Cell 12: Base Model 5 — kNN
def train_knn(X_train, X_test, y_train, y_test, n_neighbors=5, trial_id=""):
    """
    kNN OOF probability.
    """
    # Scale
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)
    
    oof_proba_train = np.zeros(len(X_train))
    oof_proba_test = np.zeros(len(X_test))
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_models = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_val = X_train_sc[train_idx], X_train_sc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = KNeighborsClassifier(n_neighbors=n_neighbors)
        model.fit(X_tr, y_tr)
        
        oof_proba_train[val_idx] = model.predict_proba(X_val)[:, 1]
        oof_proba_test += model.predict_proba(X_test_sc)[:, 1] / 5
        fold_models.append(model)
    
    return oof_proba_train, oof_proba_test, fold_models

print("kNN trainer hazır.")

kNN trainer hazır.


In [13]:
# Cell 13: Isotonic Kalibrasyon
def isotonic_calibrate_oof(oof_proba_train, oof_proba_test, y_train):
    """
    Isotonic Regresyon ile OOF olasılık calibration (train üzerinde fit).
    """
    iso_reg = IsotonicRegression(out_of_bounds='clip')
    iso_reg.fit(oof_proba_train, y_train)
    
    oof_train_calib = iso_reg.transform(oof_proba_train)
    oof_test_calib = iso_reg.transform(oof_proba_test)
    
    return oof_train_calib, oof_test_calib, iso_reg

print("Isotonic calibration hazır.")

Isotonic calibration hazır.


In [14]:
# Cell 14: Kalibre Stacking Pipeline (tek fonksiyon — 4 kolda yeniden kullanilir)

# Verilen feature seti uzerinde: 5 cesitli base OOF -> Isotonic kalibrasyon ->
# LogReg meta -> NB36 protokolu degerlendirme (optimize_threshold_8020 + bootstrap_8020_eval).
# Mevcut no_fe/leakage-dahil kol BOZULMADAN bu fonksiyona sarildi.


def _label_encode_for_nonnative(X_train, X_test):
    """Native olmayan modeller icin kategorikleri encode edip tum feature'lari sayisal yap.

    LabelEncoder train uzerinde fit edilir; testte gorulmeyen kategori sentinel'e map edilir.
    LR/kNN NaN kabul etmedigi icin sayisal bosluklar train medyani ile doldurulur.
    """
    Xtr, Xte = X_train.copy(), X_test.copy()
    cat_like_cols = [
        c for c in Xtr.columns
        if Xtr[c].dtype == object or str(Xtr[c].dtype) == 'category'
    ]

    for col in cat_like_cols:
        le = LabelEncoder()
        tr_vals = Xtr[col].astype('object').where(Xtr[col].notna(), '__NA__').astype(str)
        fit_vals = pd.concat([tr_vals, pd.Series(['__NA__'])], ignore_index=True)
        le.fit(fit_vals)
        Xtr[col] = le.transform(tr_vals)

        known = set(le.classes_)
        te_vals = Xte[col].astype('object').where(Xte[col].notna(), '__NA__').astype(str)
        te_vals = te_vals.map(lambda v: v if v in known else '__NA__')
        Xte[col] = le.transform(te_vals)

    for col in Xtr.columns:
        tr_num = pd.to_numeric(Xtr[col], errors='coerce')
        te_num = pd.to_numeric(Xte[col], errors='coerce')
        fill_value = tr_num.median()
        if pd.isna(fill_value):
            fill_value = 0.0
        Xtr[col] = tr_num.fillna(fill_value).astype(float)
        Xte[col] = te_num.fillna(fill_value).astype(float)

    return Xtr, Xte



def run_stacking_pipeline(X_train_in, X_test_in, y_train, y_test, label="arm",
                          n_bootstrap=50, verbose=True):
    """Tam kalibre-stacking + NB36 protokolu degerlendirme. Donus: dict."""
    if verbose:
        print(f"\n{'='*55}\n[{label}] Base modeller egitiliyor — X_train {X_train_in.shape}\n{'='*55}")

    cat_cols_native = [c for c in X_train_in.columns
                       if c in CAT_COLS or c in AA_COLS or X_train_in[c].dtype == object or str(X_train_in[c].dtype) == 'category']
    Xtr_le, Xte_le = _label_encode_for_nonnative(X_train_in, X_test_in)

    base_results = {}
    if verbose: print("  -> LightGBM")
    tr, te, _ = train_lgbm(Xtr_le, Xte_le, y_train, y_test);            base_results['lgbm'] = (tr, te)
    if verbose: print("  -> CatBoost (native categorical)")
    cat_in = [c for c in cat_cols_native if c in X_train_in.columns]
    tr, te, _ = train_catboost(X_train_in, X_test_in, y_train, y_test, cat_cols=cat_in); base_results['catboost'] = (tr, te)
    if verbose: print("  -> XGBoost")
    tr, te, _ = train_xgboost(Xtr_le, Xte_le, y_train, y_test);         base_results['xgboost'] = (tr, te)
    if verbose: print("  -> LogisticRegression (L2)")
    tr, te, _ = train_logistic(Xtr_le, Xte_le, y_train, y_test, penalty='l2'); base_results['logistic_l2'] = (tr, te)
    if verbose: print("  -> kNN")
    tr, te, _ = train_knn(Xtr_le, Xte_le, y_train, y_test, n_neighbors=5); base_results['knn'] = (tr, te)

    # --- Isotonic kalibrasyon (her base OOF train uzerinde fit) ---
    base_calib = {}
    for name, (otr, ote) in base_results.items():
        ctr, cte, _ = isotonic_calibrate_oof(otr, ote, y_train)
        base_calib[name] = (ctr, cte)

    # --- Meta-features (kalibre OOF) ---
    Xmeta_tr = pd.DataFrame({f'{n}_proba': base_calib[n][0] for n in base_calib})
    Xmeta_te = pd.DataFrame({f'{n}_proba': base_calib[n][1] for n in base_calib})

    # --- Meta-learner: LogisticRegression (L2, balanced) — GBM/agac DEGIL ---
    meta = LogisticRegression(penalty='l2', C=1.0, max_iter=1000,
                              random_state=SEED, class_weight='balanced')
    meta.fit(Xmeta_tr, y_train)
    proba_tr = meta.predict_proba(Xmeta_tr)[:, 1]
    proba_te = meta.predict_proba(Xmeta_te)[:, 1]

    # --- Threshold 3 mod (NB36 birebir): f1_raw / f1_8020 / mcc_8020 ---
    thr_f1_raw, f1_raw = optimize_threshold(y_train, proba_tr)
    thr_f1_8020, thr_mcc_8020 = optimize_threshold_8020(y_train, proba_tr)

    # --- %50/50 test (birincil threshold = f1_8020) ---
    y_pred_5050 = (proba_te >= thr_f1_8020).astype(int)
    m5050 = compute_all_metrics(y_test, y_pred_5050, proba_te)

    # --- %80/20 bootstrap test (f1_8020 threshold) ---
    bs = bootstrap_8020_eval(y_test, proba_te, thr_f1_8020, n_bootstrap=n_bootstrap)

    # --- Train (overfit kontrolu) ---
    y_pred_tr = (proba_tr >= thr_f1_8020).astype(int)
    mtrain = compute_all_metrics(y_train, y_pred_tr, proba_tr)

    # --- is_missing_* flag importance (LightGBM tam-fit; bu feature-set uzerinde) ---
    miss_flags = [c for c in Xtr_le.columns if str(c).startswith('is_missing_')]
    miss_fi_pct, top_miss = 0.0, []
    fi_full = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=SEED, verbose=-1)
    fi_full.fit(Xtr_le, y_train)
    fi_df = pd.DataFrame({'feature': Xtr_le.columns, 'importance': fi_full.feature_importances_})
    total_fi = fi_df['importance'].sum()
    if miss_flags and total_fi > 0:
        mfi = fi_df[fi_df['feature'].isin(miss_flags)]
        miss_fi_pct = mfi['importance'].sum() / total_fi * 100
        top_miss = mfi.sort_values('importance', ascending=False).head(5)['feature'].tolist()

    if verbose:
        print(f"  [{label}] thr_f1_8020={thr_f1_8020:.3f} mcc_thr={thr_mcc_8020:.3f} | "
              f"F1_5050={m5050['f1']:.4f} | F1_8020={bs['f1_mean']:.4f} "
              f"[{bs['f1_ci_lo']:.3f}-{bs['f1_ci_hi']:.3f}] | "
              f"gap={mtrain['f1']-m5050['f1']:+.4f} | is_missing_FI={miss_fi_pct:.1f}%")

    return {
        'label': label, 'n_features': X_train_in.shape[1],
        'thr_f1_raw': thr_f1_raw, 'thr_f1_8020': thr_f1_8020, 'thr_mcc_8020': thr_mcc_8020,
        'f1_5050': m5050['f1'], 'mcc_5050': m5050['mcc'], 'auc_roc': m5050['auc_roc'], 'auc_pr': m5050['auc_pr'],
        'prec_5050': m5050['precision'], 'rec_5050': m5050['recall'],
        'f1_8020_mean': bs['f1_mean'], 'f1_8020_std': bs['f1_std'],
        'f1_8020_ci_lo': bs['f1_ci_lo'], 'f1_8020_ci_hi': bs['f1_ci_hi'], 'f1_8020_scores': bs['f1_scores'],
        'mcc_8020': bs['mcc_mean'], 'prec_8020': bs['prec_mean'], 'rec_8020': bs['rec_mean'],
        'train_f1': mtrain['f1'], 'train_gap': mtrain['f1'] - m5050['f1'],
        'meta_model': meta, 'meta_coef': dict(zip(list(base_calib), meta.coef_[0].round(4))),
        'proba_test': proba_te, 'miss_fi_pct': miss_fi_pct, 'top_miss_flags': top_miss,
    }


print("run_stacking_pipeline() hazir — 4 kolda kullanilacak (NaN-safe CatBoost + robust encoding).")


run_stacking_pipeline() hazir — 4 kolda kullanilacak (NaN-safe CatBoost + robust encoding).


In [15]:
# --- Safety patch: CatBoost native kategorik NaN fix ---
# Bu blok Cell 15 tek basina tekrar calistirildiginda bile eski kernel tanimlarini gecerersiz kilar.
CAT_MISSING_TOKEN = "__CAT_MISSING__"


def _prepare_catboost_native(X_train, X_test, cat_cols=None):
    cat_cols = [c for c in (cat_cols or []) if c in X_train.columns]
    cat_set = set(cat_cols)
    X_tr = X_train.copy()
    X_te = X_test.copy()
    for col in X_tr.columns:
        if col in cat_set:
            X_tr[col] = X_tr[col].astype('object').where(X_tr[col].notna(), CAT_MISSING_TOKEN).astype(str)
            X_te[col] = X_te[col].astype('object').where(X_te[col].notna(), CAT_MISSING_TOKEN).astype(str)
        else:
            tr_num = pd.to_numeric(X_tr[col], errors='coerce')
            te_num = pd.to_numeric(X_te[col], errors='coerce')
            fill_value = tr_num.median()
            if pd.isna(fill_value):
                fill_value = 0.0
            X_tr[col] = tr_num.fillna(fill_value).astype(float)
            X_te[col] = te_num.fillna(fill_value).astype(float)
    return X_tr, X_te, cat_cols


def train_catboost(X_train, X_test, y_train, y_test, cat_cols=None, trial_id=""):
    X_cb_train, X_cb_test, cat_cols = _prepare_catboost_native(X_train, X_test, cat_cols)
    cat_features = cat_cols if cat_cols else []
    oof_proba_train = np.zeros(len(X_cb_train))
    oof_proba_test = np.zeros(len(X_cb_test))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_models = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_cb_train, y_train)):
        X_tr, X_val = X_cb_train.iloc[train_idx], X_cb_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model = CatBoostClassifier(
            iterations=100,
            learning_rate=0.1,
            random_state=SEED,
            cat_features=cat_features,
            verbose=False
        )
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
        oof_proba_train[val_idx] = model.predict_proba(X_val)[:, 1]
        oof_proba_test += model.predict_proba(X_cb_test)[:, 1] / 5
        fold_models.append(model)
    return oof_proba_train, oof_proba_test, fold_models


def _label_encode_for_nonnative(X_train, X_test):
    Xtr, Xte = X_train.copy(), X_test.copy()
    cat_like_cols = [c for c in Xtr.columns if Xtr[c].dtype == object or str(Xtr[c].dtype) == 'category']
    for col in cat_like_cols:
        le = LabelEncoder()
        tr_vals = Xtr[col].astype('object').where(Xtr[col].notna(), '__NA__').astype(str)
        fit_vals = pd.concat([tr_vals, pd.Series(['__NA__'])], ignore_index=True)
        le.fit(fit_vals)
        Xtr[col] = le.transform(tr_vals)
        known = set(le.classes_)
        te_vals = Xte[col].astype('object').where(Xte[col].notna(), '__NA__').astype(str)
        te_vals = te_vals.map(lambda v: v if v in known else '__NA__')
        Xte[col] = le.transform(te_vals)
    for col in Xtr.columns:
        tr_num = pd.to_numeric(Xtr[col], errors='coerce')
        te_num = pd.to_numeric(Xte[col], errors='coerce')
        fill_value = tr_num.median()
        if pd.isna(fill_value):
            fill_value = 0.0
        Xtr[col] = tr_num.fillna(fill_value).astype(float)
        Xte[col] = te_num.fillna(fill_value).astype(float)
    return Xtr, Xte

# Cell 15: FE (Grantham/BLOSUM62) + Robust set + 4 Kolu Calistir (no_fe/with_fe × dahil/robust)
# Spec madde 4 (FE ablasyon) + madde 5 (leakage robust) GERCEKTEN kosulan kollar.
# FE ve make_robust burada tanimli (ayri hucre kaymasina bagimli degil).
import time
from src.features import GRANTHAM, BLOSUM62  # NB36 Cell 12 ile birebir gomulu tablolar

def apply_fe(X):
    """EK meta-prediktörler + AA fizikokimyasal FE (NB36 Cell 12 ile BIREBIR)."""
    X = X.copy()
    ek_present = [c for c in EK_COLS if c in X.columns]
    if ek_present:
        ek_vals = X[ek_present]
        X['ek_mean_all'] = ek_vals.mean(axis=1)
        X['ek_max'] = ek_vals.max(axis=1)
        X['ek_min'] = ek_vals.min(axis=1)
        X['ek_std'] = ek_vals.std(axis=1)
        X['ek_range'] = X['ek_max'] - X['ek_min']
        if 'EK_7' in X.columns and 'EK_1' in X.columns:
            X['ek_7_minus_1'] = X['EK_7'] - X['EK_1']   # disagreement
        if 'EK_7' in X.columns and 'EK_8' in X.columns:
            X['ek_7_minus_8'] = X['EK_7'] - X['EK_8']
        X['ek_n_positive'] = (ek_vals > 0).sum(axis=1)
    if 'AA_1' in X.columns and 'AA_2' in X.columns:
        aa1 = X['AA_1'].astype(str).str.upper().str.strip()
        aa2 = X['AA_2'].astype(str).str.upper().str.strip()
        X['grantham_distance'] = [GRANTHAM.get((a, b), np.nan) if a != b else 0
                                  for a, b in zip(aa1, aa2)]
        X['blosum62_score'] = [BLOSUM62.get((a, b), np.nan) for a, b in zip(aa1, aa2)]
        X['blosum62_ref_self'] = [BLOSUM62.get((a, a), np.nan) for a in aa1]
        X['blosum62_delta'] = X['blosum62_ref_self'] - X['blosum62_score']
        X['grantham_cat'] = pd.cut(X['grantham_distance'], bins=[-1, 50, 100, 150, 300],
                                   labels=[0, 1, 2, 3]).astype(float)
        X['is_synonymous'] = (aa1 == aa2).astype(int)
    return X

def make_robust(X):
    """AL_MISSINGNESS_LEAKAGE_RISK sutunlari + tum is_missing_* flaglerini cikar (robust)."""
    leak = [c for c in AL_MISSINGNESS_LEAKAGE_RISK if c in X.columns]
    leak_flags = [get_missing_mask_col_name(c) for c in AL_MISSINGNESS_LEAKAGE_RISK
                  if get_missing_mask_col_name(c) in X.columns]
    all_miss_flags = [c for c in X.columns if str(c).startswith('is_missing_')]
    drop = list(set(leak + leak_flags + all_miss_flags))
    return X.drop(columns=drop, errors='ignore'), drop

# --- with_fe feature seti uret (no_fe = X_train_imp; Cell 5'te hazir) ---
X_train_fe_raw = apply_fe(X_train_imp)
X_test_fe_raw  = apply_fe(X_test_imp)
# FE'nin urettigi yeni sayisal NaN'lari train-medyani ile doldur (sadece yeni FE sutunlari)
fe_new_cols = [c for c in X_train_fe_raw.columns if c not in X_train_imp.columns]
fe_numeric = [c for c in fe_new_cols if X_train_fe_raw[c].dtype != object]
if fe_numeric:
    _fe_imp = SimpleImputer(strategy='median')
    X_train_fe_raw[fe_numeric] = _fe_imp.fit_transform(X_train_fe_raw[fe_numeric])
    X_test_fe_raw[fe_numeric]  = _fe_imp.transform(X_test_fe_raw[fe_numeric])
print(f"FE yeni sutunlar ({len(fe_new_cols)}): {fe_new_cols}")
print(f"no_fe feature={X_train_imp.shape[1]} | with_fe feature={X_train_fe_raw.shape[1]}")

# --- Robust setler ---
X_nofe_rob_tr, _nofe_rob_drop = make_robust(X_train_imp)
X_nofe_rob_te, _              = make_robust(X_test_imp)
X_wfe_rob_tr,  _wfe_rob_drop  = make_robust(X_train_fe_raw)
X_wfe_rob_te,  _              = make_robust(X_test_fe_raw)
print(f"Robust drop: no_fe={len(_nofe_rob_drop)}, with_fe={len(_wfe_rob_drop)}")

# --- 4 kol ---
ARMS = [
    ("no_fe_dahil",    X_train_imp,    X_test_imp),     # BIRINCIL (korunan davranis)
    ("with_fe_dahil",  X_train_fe_raw, X_test_fe_raw),
    ("no_fe_robust",   X_nofe_rob_tr,  X_nofe_rob_te),
    ("with_fe_robust", X_wfe_rob_tr,   X_wfe_rob_te),
]
arm_results = OrderedDict()
_t0 = time.time()
for _label, _Xtr, _Xte in ARMS:
    arm_results[_label] = run_stacking_pipeline(_Xtr, _Xte, y_train, y_test, label=_label)
print(f"\n4 kol tamamlandi ({time.time()-_t0:.1f}s).")

# --- Birincil kol geri-uyum referanslari ---
_primary = arm_results['no_fe_dahil']
metrics_test_5050 = {'f1': _primary['f1_5050'], 'auc_roc': _primary['auc_roc'],
                     'auc_pr': _primary['auc_pr'], 'precision': _primary['prec_5050'],
                     'recall': _primary['rec_5050']}
mean_f1_8020   = _primary['f1_8020_mean']
ci_lower_8020  = _primary['f1_8020_ci_lo']
ci_upper_8020  = _primary['f1_8020_ci_hi']
f1_scores_8020 = _primary['f1_8020_scores']
thr_f1_8020    = _primary['thr_f1_8020']
thr_f1_raw     = _primary['thr_f1_raw']
thr_mcc_8020   = _primary['thr_mcc_8020']
y_proba_meta_test = _primary['proba_test']
metrics_train  = {'f1': _primary['train_f1']}
meta_model     = _primary['meta_model']


FE yeni sutunlar (14): ['ek_mean_all', 'ek_max', 'ek_min', 'ek_std', 'ek_range', 'ek_7_minus_1', 'ek_7_minus_8', 'ek_n_positive', 'grantham_distance', 'blosum62_score', 'blosum62_ref_self', 'blosum62_delta', 'grantham_cat', 'is_synonymous']
no_fe feature=426 | with_fe feature=440
Robust drop: no_fe=149, with_fe=149

[no_fe_dahil] Base modeller egitiliyor — X_train (2344, 426)
  -> LightGBM
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[22]	valid_0's binary_logloss: 0.451023
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[37]	valid_0's binary_logloss: 0.427096
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[25]	valid_0's binary_logloss: 0.444431
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[32]	valid_0's binary_logloss: 0.465942
Training until validation scores don't improve for 10 rounds
Ea

In [16]:
# Cell 16: 4-Kol Karsilastirma Tablosu (%80/20 F1 + MCC + precision + recall)
comparison_df = pd.DataFrame([
    {
        'arm': r['label'],
        'n_feat': r['n_features'],
        'F1_8020': round(r['f1_8020_mean'], 4),
        'F1_8020_CI': f"[{r['f1_8020_ci_lo']:.3f}, {r['f1_8020_ci_hi']:.3f}]",
        'MCC_8020': round(r['mcc_8020'], 4),
        'Prec_8020': round(r['prec_8020'], 4),
        'Rec_8020': round(r['rec_8020'], 4),
        'F1_5050': round(r['f1_5050'], 4),
        'AUC_ROC': round(r['auc_roc'], 4),
        'train_gap': round(r['train_gap'], 4),
        'is_missing_FI%': round(r['miss_fi_pct'], 1),
        'thr_f1_8020': round(r['thr_f1_8020'], 3),
    }
    for r in arm_results.values()
])

print("=" * 90)
print("4-KOL KARSILASTIRMA (Birincil metrik: %80/20 bootstrap pathogenic-F1)")
print("=" * 90)
print(comparison_df.to_string(index=False))

# Kazananlari belirle
best_arm = comparison_df.loc[comparison_df['F1_8020'].idxmax(), 'arm']
fe_dahil = comparison_df[comparison_df['arm'] == 'with_fe_dahil']['F1_8020'].values[0]
nofe_dahil = comparison_df[comparison_df['arm'] == 'no_fe_dahil']['F1_8020'].values[0]
fe_delta = fe_dahil - nofe_dahil
rob_dahil = comparison_df[comparison_df['arm'] == 'no_fe_robust']['F1_8020'].values[0]
rob_delta = rob_dahil - nofe_dahil

print(f"\n→ EN IYI KOL (%80/20 F1): {best_arm}")
print(f"→ FE etkisi (with_fe - no_fe, leakage-dahil): {fe_delta:+.4f}  "
      f"({'FE FAYDALI' if fe_delta > 0.005 else 'FE ZARARLI/NOTR — NB36 ablasyonu teyit' if fe_delta < -0.005 else 'FARK YOK'})")
print(f"→ Robust etkisi (robust - dahil, no_fe): {rob_delta:+.4f}  "
      f"({'robust kayipsiz' if rob_delta > -0.01 else 'robust kayipli (leakage train avantaji)'})")

# Kaydet
results_dir = os.path.join(PROJECT_ROOT, 'results/v20_master_calibrated_stacking')
os.makedirs(results_dir, exist_ok=True)
comparison_df.to_csv(os.path.join(results_dir, 'arms_comparison.csv'), index=False)
print(f"\nKaydedildi: {results_dir}/arms_comparison.csv")


4-KOL KARSILASTIRMA (Birincil metrik: %80/20 bootstrap pathogenic-F1)
           arm  n_feat  F1_8020     F1_8020_CI  MCC_8020  Prec_8020  Rec_8020  F1_5050  AUC_ROC  train_gap  is_missing_FI%  thr_f1_8020
   no_fe_dahil     426   0.5776 [0.507, 0.647]    0.4655     0.4615    0.7754   0.8321   0.8500     0.0026             0.9        0.620
 with_fe_dahil     440   0.5853 [0.512, 0.669]    0.4735     0.4850    0.7436   0.8184   0.8503     0.0005             0.9        0.645
  no_fe_robust     277   0.5890 [0.502, 0.651]    0.4774     0.5086    0.7046   0.7963   0.8559    -0.0194             0.0        0.680
with_fe_robust     291   0.5782 [0.507, 0.675]    0.4633     0.5003    0.6903   0.7900   0.8556    -0.0071             0.0        0.680

→ EN IYI KOL (%80/20 F1): no_fe_robust
→ FE etkisi (with_fe - no_fe, leakage-dahil): +0.0077  (FE FAYDALI)
→ Robust etkisi (robust - dahil, no_fe): +0.0114  (robust kayipsiz)

Kaydedildi: /Users/tefe/teknofest_model/teknofest_model/results/v20_maste

In [17]:
# Cell 17: is_missing_* Flag Importance ve Leakage Uyarisi (spec madde 5)
print("=" * 70)
print("LEAKAGE PROBE: is_missing_* flag importance (LightGBM tam-fit)")
print("=" * 70)

LEAK_ALARM_PCT = 15.0  # NB36 esigi
for label in ['no_fe_dahil', 'with_fe_dahil']:
    r = arm_results[label]
    pct = r['miss_fi_pct']
    alarm = pct > LEAK_ALARM_PCT
    print(f"\n[{label}] is_missing_* toplam importance: {pct:.1f}%")
    print(f"  Top is_missing flagleri: {r['top_miss_flags']}")
    if alarm:
        print(f"  *** LEAKAGE UYARISI: is_missing flagleri importance'ta baskin (>{LEAK_ALARM_PCT}%)!")
        print(f"      Model 'sutun bossa pathogenic' kisayolunu ogreniyor olabilir.")
        print(f"      → Robust kol (leakage-haric) yarisma teslimi icin yedek tutulmali.")
    else:
        print(f"  is_missing importance kontrollu (<={LEAK_ALARM_PCT}%) — leakage baskin degil.")

# Robust kollarda is_missing flag YOK (cikarildi) — teyit
print("\nRobust kollarda is_missing_* flag sayisi (0 olmali):")
for label in ['no_fe_robust', 'with_fe_robust']:
    print(f"  [{label}] is_missing_FI% = {arm_results[label]['miss_fi_pct']:.1f}% (cikarildigi icin 0 beklenir)")

# Genel leakage karari
worst_pct = max(arm_results['no_fe_dahil']['miss_fi_pct'], arm_results['with_fe_dahil']['miss_fi_pct'])
LEAKAGE_VERDICT = ("YUKSEK — robust varyant zorunlu yedek" if worst_pct > LEAK_ALARM_PCT
                   else "DUSUK — leakage'a yaslanma sinirli")
print(f"\n→ GENEL LEAKAGE KARARI: {LEAKAGE_VERDICT} (max is_missing_FI={worst_pct:.1f}%)")


LEAKAGE PROBE: is_missing_* flag importance (LightGBM tam-fit)

[no_fe_dahil] is_missing_* toplam importance: 0.9%
  Top is_missing flagleri: ['is_missing_AL_16', 'is_missing_AL_1', 'is_missing_AL_183', 'is_missing_AL_214', 'is_missing_AL_239']
  is_missing importance kontrollu (<=15.0%) — leakage baskin degil.

[with_fe_dahil] is_missing_* toplam importance: 0.9%
  Top is_missing flagleri: ['is_missing_AL_1', 'is_missing_AL_16', 'is_missing_AL_183', 'is_missing_AL_286', 'is_missing_AL_187']
  is_missing importance kontrollu (<=15.0%) — leakage baskin degil.

Robust kollarda is_missing_* flag sayisi (0 olmali):
  [no_fe_robust] is_missing_FI% = 0.0% (cikarildigi icin 0 beklenir)
  [with_fe_robust] is_missing_FI% = 0.0% (cikarildigi icin 0 beklenir)

→ GENEL LEAKAGE KARARI: DUSUK — leakage'a yaslanma sinirli (max is_missing_FI=0.9%)


In [18]:
# Cell 18: Görselleştirmeler (4-kol karsilastirma + birincil kol detayi)
results_dir = os.path.join(PROJECT_ROOT, 'results/v20_master_calibrated_stacking')
os.makedirs(results_dir, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
arm_labels = list(arm_results.keys())
arm_colors = ['#2980b9', '#27ae60', '#e67e22', '#8e44ad']

# 1. 4-kol %80/20 F1 (bar + CI)
means = [arm_results[a]['f1_8020_mean'] for a in arm_labels]
ci_lo = [arm_results[a]['f1_8020_mean'] - arm_results[a]['f1_8020_ci_lo'] for a in arm_labels]
ci_hi = [arm_results[a]['f1_8020_ci_hi'] - arm_results[a]['f1_8020_mean'] for a in arm_labels]
axes[0, 0].bar(range(len(arm_labels)), means, yerr=[ci_lo, ci_hi], capsize=5, color=arm_colors)
axes[0, 0].set_xticks(range(len(arm_labels)))
axes[0, 0].set_xticklabels(arm_labels, rotation=20, ha='right', fontsize=8)
axes[0, 0].set_ylabel('%80/20 Pathogenic F1')
axes[0, 0].set_title('4-Kol Karsilastirma (%80/20 F1 + 95% CI)')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# 2. Bootstrap F1 dagilimi (birincil kol)
_prim = arm_results['no_fe_dahil']
axes[0, 1].hist(_prim['f1_8020_scores'], bins=20, edgecolor='black', alpha=0.7, color='#2980b9')
axes[0, 1].axvline(_prim['f1_8020_mean'], color='r', linestyle='--', linewidth=2,
                   label=f"Mean: {_prim['f1_8020_mean']:.3f}")
axes[0, 1].axvline(_prim['f1_8020_ci_lo'], color='g', linestyle=':', linewidth=2, label='95% CI')
axes[0, 1].axvline(_prim['f1_8020_ci_hi'], color='g', linestyle=':', linewidth=2)
axes[0, 1].set_xlabel('F1'); axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Bootstrap F1 Dagilimi — no_fe_dahil (birincil)')
axes[0, 1].legend()

# 3. is_missing_* flag importance (4 kol)
miss_pcts = [arm_results[a]['miss_fi_pct'] for a in arm_labels]
axes[1, 0].bar(range(len(arm_labels)), miss_pcts, color=arm_colors)
axes[1, 0].axhline(15.0, color='r', linestyle='--', label='Leakage alarm esigi (%15)')
axes[1, 0].set_xticks(range(len(arm_labels)))
axes[1, 0].set_xticklabels(arm_labels, rotation=20, ha='right', fontsize=8)
axes[1, 0].set_ylabel('is_missing_* toplam importance (%)')
axes[1, 0].set_title('Leakage Probe: is_missing Flag Importance')
axes[1, 0].legend()

# 4. Confusion matrix (birincil kol, %50/50)
y_proba_prim = _prim['proba_test']
y_pred_prim = (y_proba_prim >= _prim['thr_f1_8020']).astype(int)
cm = confusion_matrix(y_test, y_pred_prim)
axes[1, 1].imshow(cm, cmap='Blues')
axes[1, 1].set_xlabel('Predicted'); axes[1, 1].set_ylabel('Actual')
axes[1, 1].set_title('Confusion Matrix — no_fe_dahil (%50/50)')
axes[1, 1].set_xticks([0, 1]); axes[1, 1].set_yticks([0, 1])
for i in range(2):
    for j in range(2):
        axes[1, 1].text(j, i, str(cm[i, j]), ha='center', va='center',
                        color='white' if cm[i, j] > cm.max()/2 else 'black')

plt.tight_layout()
fig_path = os.path.join(results_dir, 'fig_nb37_arms.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Görsel: {fig_path}")
plt.close()


Görsel: /Users/tefe/teknofest_model/teknofest_model/results/v20_master_calibrated_stacking/fig_nb37_arms.png


In [19]:
# Cell 19: PDF Rapor (4-kol kiyas tablosu + leakage + NB14/NB36 kiyas)
class CalibStackingReport(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 14)
        self.cell(0, 10, "NB37: MASTER Panel - Kalibre Stacking Raporu", 0, 1, "C")
        self.set_font("Helvetica", "", 9)
        self.cell(0, 5, f"SEED={SEED} | TEST_SIZE={TEST_SIZE} | 4 kol x 5 cesitli base + Isotonic", 0, 1, "C")
        self.ln(3)

    def section_title(self, title):
        self.set_font("Helvetica", "B", 12)
        self.set_fill_color(41, 128, 185)
        self.set_text_color(255, 255, 255)
        self.cell(0, 8, f"  {title}", 0, 1, "L", fill=True)
        self.set_text_color(0, 0, 0)
        self.ln(2)

    def body_text(self, text):
        self.set_font("Helvetica", "", 9)
        self.multi_cell(0, 5, text)
        self.ln(2)

    def add_table(self, headers, rows, col_widths):
        self.set_font("Helvetica", "B", 7)
        self.set_fill_color(52, 73, 94)
        self.set_text_color(255, 255, 255)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, str(h), 1, 0, "C", fill=True)
        self.ln()
        self.set_font("Helvetica", "", 7)
        self.set_text_color(0, 0, 0)
        for j, row in enumerate(rows):
            self.set_fill_color(236, 240, 241) if j % 2 == 0 else self.set_fill_color(255, 255, 255)
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), 1, 0, "C", fill=True)
            self.ln()
        self.ln(3)


pdf = CalibStackingReport()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# --- KIYAS bolumu (en basta) ---
pdf.section_title("0. KIYAS: NB14 (%50/50 yanilsamasi) vs NB36 vs NB37")
_p = arm_results['no_fe_dahil']
pdf.body_text(
    "NB14'un %50/50 F1~0.89'u TRAIN-DENGESI YANILSAMASIDIR: egitim ~%73 pathogenic; final "
    "test ise ~%80 BENIGN. Train dengesinde olculen F1 final dagilimi YANSITMAZ. Gercek "
    "metrik = %80/20 bootstrap pathogenic-F1.\n\n"
    "Bu notebook (NB37) ayni kalibre+cesitli stacking'i HEM %50/50 HEM %80/20 raporlar."
)
kiyas_headers = ["Notebook", "Yontem", "%50/50 F1", "%80/20 F1 (gercek)"]
kiyas_rows = [
    ["NB14", "OOF stack (7 base), %50/50 only", "~0.89", "OLCULMEDI (eksik)"],
    ["NB36", "Baseline (no calib stack)", "~0.60", "~0.60 (rapor edildi)"],
    ["NB37", "Kalibre+cesitli stack (no_fe_dahil)",
     f"{_p['f1_5050']:.4f}", f"{_p['f1_8020_mean']:.4f} [{_p['f1_8020_ci_lo']:.3f}-{_p['f1_8020_ci_hi']:.3f}]"],
]
pdf.add_table(kiyas_headers, kiyas_rows, [30, 70, 30, 55])

# --- 4-kol kiyas tablosu (4 kol x %80/20 F1 + MCC + prec + rec) ---
pdf.section_title("1. 4-KOL KARSILASTIRMA (no_fe/with_fe x dahil/robust)")
pdf.body_text("Birincil metrik = %80/20 bootstrap pathogenic-F1. FE ve robust kollari GERCEKTEN egitilir.")
arm_headers = ["Kol", "n_feat", "F1_8020", "CI_95", "MCC_8020", "Prec_8020", "Rec_8020", "F1_5050", "gap", "missFI%"]
arm_rows = []
for r in arm_results.values():
    arm_rows.append([
        r['label'], r['n_features'], f"{r['f1_8020_mean']:.4f}",
        f"[{r['f1_8020_ci_lo']:.2f},{r['f1_8020_ci_hi']:.2f}]",
        f"{r['mcc_8020']:.3f}", f"{r['prec_8020']:.3f}", f"{r['rec_8020']:.3f}",
        f"{r['f1_5050']:.3f}", f"{r['train_gap']:+.3f}", f"{r['miss_fi_pct']:.1f}",
    ])
pdf.add_table(arm_headers, arm_rows, [30, 14, 20, 26, 20, 20, 20, 18, 16, 16])

# Kazanan kol ve FE/robust delta yorumu
best_arm = max(arm_results.values(), key=lambda r: r['f1_8020_mean'])['label']
fe_delta = arm_results['with_fe_dahil']['f1_8020_mean'] - arm_results['no_fe_dahil']['f1_8020_mean']
rob_delta = arm_results['no_fe_robust']['f1_8020_mean'] - arm_results['no_fe_dahil']['f1_8020_mean']
fe_verdict = ("FE FAYDALI" if fe_delta > 0.005 else
              "FE ZARARLI/NOTR (NB36 ablasyonunu TEYIT)" if fe_delta < -0.005 else "FE FARK YOK")
pdf.body_text(
    f"EN IYI KOL (%80/20 F1): {best_arm}\n"
    f"FE etkisi (with_fe - no_fe, dahil): {fe_delta:+.4f} -> {fe_verdict}\n"
    f"Robust etkisi (robust - dahil, no_fe): {rob_delta:+.4f} "
    f"({'robust kayipsiz, guvenli teslim' if rob_delta > -0.01 else 'robust kayipli (leakage train avantaji)'})"
)

# --- Leakage analizi ---
pdf.section_title("2. Leakage Analizi (is_missing_* flag importance)")
worst_pct = max(arm_results['no_fe_dahil']['miss_fi_pct'], arm_results['with_fe_dahil']['miss_fi_pct'])
leak_verdict = ("YUKSEK - robust varyant zorunlu yedek" if worst_pct > 15.0
                else "DUSUK - leakage'a yaslanma sinirli")
pdf.body_text(
    f"is_missing_* flag importance (dahil kollar):\n"
    f"  no_fe_dahil  : {arm_results['no_fe_dahil']['miss_fi_pct']:.1f}%  "
    f"(top: {arm_results['no_fe_dahil']['top_miss_flags'][:3]})\n"
    f"  with_fe_dahil: {arm_results['with_fe_dahil']['miss_fi_pct']:.1f}%\n\n"
    f"Robust kollarda is_missing flagleri CIKARILDI (importance=0 beklenir).\n\n"
    f"GENEL LEAKAGE KARARI: {leak_verdict} (max is_missing_FI={worst_pct:.1f}%, alarm esigi %15).\n"
    f"Yarisma test eksiklik oruntu su bilinmedigi icin robust kol teslim yedegi olarak tutulmali."
)

# --- Yontem ---
pdf.section_title("3. Yontem")
pdf.body_text(
    "Base Models (5 cesitli): LightGBM, CatBoost (native cat), XGBoost, LogisticRegression(L2), kNN.\n"
    "OOF: 5-fold StratifiedKFold. Isotonic kalibrasyon: her base OOF train uzerinde fit.\n"
    "Meta-Learner: LogisticRegression(C=1.0, L2, class_weight=balanced) -- GBM/agac meta KULLANILMADI.\n"
    "Threshold: 3 mod (f1_raw / f1_8020 / mcc_8020); birincil = f1_8020 (%80/20 havuzda secildi).\n"
    "Missing: M3 (>50% NaN icin is_missing_* flag + medyan imputation, orijinal korunur).\n"
    "FE (with_fe kollari): EK meta (mean/max/min/std/range/disagreement/n_pos) + Grantham/BLOSUM62 "
    "(src/features.py tablolari) -- NB36 Cell 12 ile birebir."
)

# --- Gorsel ---
pdf.add_page()
pdf.section_title("4. Gorseller")
if os.path.exists(fig_path):
    pdf.image(fig_path, x=8, w=195)

report_path = os.path.join(REPORTS_DIR, 'NB37_master_calibrated_stacking_report.pdf')
pdf.output(report_path)
print(f"PDF rapor: {report_path}")

# Birincil ozet CSV (geri uyum)
results_summary = pd.DataFrame({
    'Metric': ['best_arm', 'thr_f1_8020(no_fe_dahil)', 'F1_5050(no_fe_dahil)',
               'F1_8020(no_fe_dahil)', 'FE_delta', 'robust_delta', 'max_is_missing_FI%'],
    'Value': [best_arm, f"{_p['thr_f1_8020']:.4f}", f"{_p['f1_5050']:.4f}",
              f"{_p['f1_8020_mean']:.4f}", f"{fe_delta:+.4f}", f"{rob_delta:+.4f}", f"{worst_pct:.1f}"]
})
results_summary.to_csv(os.path.join(results_dir, 'stacking_summary.csv'), index=False)
print(f"Ozet CSV: {results_dir}/stacking_summary.csv")


PDF rapor: /Users/tefe/teknofest_model/teknofest_model/reports/NB37_master_calibrated_stacking_report.pdf
Ozet CSV: /Users/tefe/teknofest_model/teknofest_model/results/v20_master_calibrated_stacking/stacking_summary.csv


In [20]:
# Cell 20: Final Özet
print("\n" + "=" * 70)
print("NB37 TAMAMLANDI: MASTER Panel — Kalibre Stacking (4 kol)")
print("=" * 70)
print("Base Models: 5 cesitli (LightGBM, CatBoost, XGBoost, LogisticRegression, kNN)")
print("Kalibrasyon: IsotonicRegression | Meta: LogisticRegression (L2, balanced)")
print("\n4-KOL %80/20 F1 (birincil metrik):")
for r in arm_results.values():
    print(f"  {r['label']:16s} F1_8020={r['f1_8020_mean']:.4f} "
          f"[{r['f1_8020_ci_lo']:.3f}-{r['f1_8020_ci_hi']:.3f}] | "
          f"MCC={r['mcc_8020']:.3f} P={r['prec_8020']:.3f} R={r['rec_8020']:.3f} | "
          f"gap={r['train_gap']:+.3f} | missFI={r['miss_fi_pct']:.1f}%")

print(f"\nEN IYI KOL: {best_arm}")
print(f"FE etkisi (with_fe-no_fe): {fe_delta:+.4f} ({fe_verdict})")
print(f"Robust etkisi (robust-dahil): {rob_delta:+.4f}")
print(f"Leakage karari: {leak_verdict}")
print("\nCiktilar:")
print(f"  • {results_dir}/arms_comparison.csv")
print(f"  • {results_dir}/stacking_summary.csv")
print(f"  • {fig_path}")
print(f"  • {report_path}")
print("\nNotebook basariyla tamamlandi (calistirma kullaniciya birakildi).")



NB37 TAMAMLANDI: MASTER Panel — Kalibre Stacking (4 kol)
Base Models: 5 cesitli (LightGBM, CatBoost, XGBoost, LogisticRegression, kNN)
Kalibrasyon: IsotonicRegression | Meta: LogisticRegression (L2, balanced)

4-KOL %80/20 F1 (birincil metrik):
  no_fe_dahil      F1_8020=0.5776 [0.507-0.647] | MCC=0.465 P=0.462 R=0.775 | gap=+0.003 | missFI=0.9%
  with_fe_dahil    F1_8020=0.5853 [0.512-0.669] | MCC=0.473 P=0.485 R=0.744 | gap=+0.000 | missFI=0.9%
  no_fe_robust     F1_8020=0.5890 [0.502-0.651] | MCC=0.477 P=0.509 R=0.705 | gap=-0.019 | missFI=0.0%
  with_fe_robust   F1_8020=0.5782 [0.507-0.675] | MCC=0.463 P=0.500 R=0.690 | gap=-0.007 | missFI=0.0%

EN IYI KOL: no_fe_robust
FE etkisi (with_fe-no_fe): +0.0077 (FE FAYDALI)
Robust etkisi (robust-dahil): +0.0114
Leakage karari: DUSUK - leakage'a yaslanma sinirli

Ciktilar:
  • /Users/tefe/teknofest_model/teknofest_model/results/v20_master_calibrated_stacking/arms_comparison.csv
  • /Users/tefe/teknofest_model/teknofest_model/results/v20_m